# Hardware patches don't belong in the monitor

Companion notebook for the [Progressive Testing](https://gtcloudrobotics.com/progressive-testing/) page, §5.

Setup: same satellite, same two redundant temperature sensors. You've taken the monitor from SIL through HIL and onto the bench. Each integration step turned up a real quirk — sensor A needs 50 ms of warmup or its first reads are garbage, sensor B's firmware drops `-127` once in a while, hardware rev B uses a slower ADC. You patched each one where you found it. A week later, the monitor looks nothing like the clean version you started with.

The behavior is still correct. The architecture isn't.

This notebook walks through:

1. Reproducing the patched-up monitor.
2. Seeing why "it works" is the trap.
3. Pulling each hardware quirk into a sensor adapter — the fix from §5 of the reading, made executable.
4. **Your job:** writing the SIL fakes that mimic the *real* hardware failure modes, so the adapter pattern stays defended on every CI run.

The reading already shows the adapter implementation. Your work here is the *test fixtures* that prove the adapters absorb what they're supposed to.

## 1. The hardware-shaped monitor

Three patches, one monitor. Each was a reasonable response to something the bench taught us. Together they have eaten the architecture.

In [ ]:
import time

class ThermalMonitor:
    def __init__(self, a, b, max_c=80.0, rev='A'):
        self.a, self.b = a, b
        self.max_c = max_c
        self.rev = rev
        self.boot = time.time()
        self.warmed = False
        self.last_b = 0.0
        self.state = 'NOMINAL'

    def tick(self):
        # Sensor A: warmup or first reads are garbage (issue #847)
        if not self.warmed and time.time() - self.boot < 0.05:
            return
        self.warmed = True
        a = self.a.read()
        b = self.b.read()
        # Sensor B firmware drops -127 occasionally — vendor fix "next quarter"
        if b == -127:
            b = self.last_b
        self.last_b = b
        # Hardware rev B uses a slower ADC; widen the alert hysteresis
        threshold = self.max_c + 1.0 if self.rev == 'B' else self.max_c
        avg = (a + b) / 2
        if avg > threshold:
            self.state = 'THERMAL_SAFE_MODE'
        else:
            self.state = 'NOMINAL'

## 2. It works — that's the trap

Run the monitor against a benign trace. Sensors stay healthy, no `-127` glitches, A's warmup is invisible because nothing bad happens during it. The monitor stays `NOMINAL` end to end. Green check, ship it.

That's the trap: **behavioral correctness is hiding architectural rot**. The monitor passes its test, the satellite stays cool, and the codebase quietly gets harder to change.

In [1]:
class FakeSensor:
    def __init__(self, readings):
        self.readings = list(readings); self.i = 0
    def read(self):
        v = self.readings[min(self.i, len(self.readings) - 1)]
        self.i += 1
        return v

a = FakeSensor([42.0, 42.1, 42.2, 42.3])
b = FakeSensor([43.0, 43.1, 43.2, 43.3])
monitor = ThermalMonitor(a, b)

time.sleep(0.06)  # past the warmup window so tick() actually runs the threshold check
for t in range(4):
    monitor.tick()
    print(f't={t}.0s  state={monitor.state}')

t=0.0s  state=NOMINAL
t=1.0s  state=NOMINAL
t=2.0s  state=NOMINAL
t=3.0s  state=NOMINAL


## 3. What's actually broken

Behaviorally the monitor is fine. Architecturally it's a mess. Count the things `tick` knows that it should not:

- That sensor A is a *device with a warmup period* — not just something with a `.read()`.
- That `-127` is a sentinel sensor B's firmware emits as a glitch — a vendor implementation detail.
- That a specific hardware revision changes the right alert threshold.

Each of those is a leak across an interface that [system design §2](https://gtcloudrobotics.com/system-design/#2-interfaces-are-the-real-architecture) tried to keep clean. The consequences compound:

- Swap sensor A's driver for one that doesn't need a warmup → dead code in the monitor, with no obvious owner.
- Bring in a third sensor with its own quirks → another branch in `tick`.
- Port to a new platform → `rev` grows another value, the threshold logic grows another arm.

And worst: the `FakeSensor` from cell 4 doesn't model warmups, `-127` glitches, or rev-B behavior. So the failure modes you learned about on hardware aren't reachable in the test suite. The safety net has quietly stopped covering the code that actually ships.

## 4. The fix from §5 of the reading

Each hardware quirk moves out of `ThermalMonitor.tick` and into a thin sensor adapter. Each adapter's only job is to present a clean `.read()` while absorbing the device's quirks internally:

- `SensorA` wraps the real device and waits out the 50 ms warmup on the first read.
- `SensorB` wraps the real device and substitutes the last valid reading whenever the firmware emits its `-127` glitch.
- The rev-B threshold tweak goes wherever revision-aware system config goes — outside the monitor, not our concern here.

`ThermalMonitorV2` is what we want back: averaging two readings, threshold-checking, nothing else.

The cell below is the lesson from §5 of the reading, made executable. **Nothing to fill in here** — the work starts in the next section.

In [ ]:
class SensorA:
    """Adapter: absorbs the 50 ms post-init warmup."""
    def __init__(self, device):
        self._device = device
        self.warm = False
    def read(self):
        if not self.warm:
            time.sleep(0.05)
            self.warm = True
        return self._device.read()

class SensorB:
    """Adapter: hides the firmware's -127 glitch by holding the last valid reading."""
    def __init__(self, device):
        self._device = device
        self.last_valid = 0.0
    def read(self):
        v = self._device.read()
        if v == -127:
            return self.last_valid
        self.last_valid = v
        return v

class ThermalMonitorV2:
    def __init__(self, a, b, max_c=80.0):
        self.a, self.b, self.max_c = a, b, max_c
        self.state = 'NOMINAL'
    def tick(self):
        avg = (self.a.read() + self.b.read()) / 2
        if avg > self.max_c:
            self.state = 'THERMAL_SAFE_MODE'
        else:
            self.state = 'NOMINAL'

## 5. Your turn: prove the adapters work

The adapter pattern only earns its keep if the absorbed behavior is *tested*. Without a SIL fixture that misbehaves the way the real hardware does, the adapter could be quietly broken on the next commit and nobody would notice until the next bench run.

Below are two stubbed-out "real" devices. Each one should reproduce a failure mode the bench taught us:

- `GarbageOnFirstReadDevice` — returns nonsense (`9999.9`) if `read()` is called less than 50 ms after the device was instantiated. After that, it advances through its scripted readings normally. (This is what sensor A's real driver does when polled before warmup completes.)
- `GlitchingDevice` — emits `-127` on its *second* call (when `self.i == 1`); otherwise advances through its scripted readings normally. (This is sensor B's firmware glitch.)

The assertions below test each adapter *directly*: pull four readings out of each adapter and check that the failure modes never leak through. That's deliberate — testing the adapter's contract is more honest than running the whole monitor and hoping the symptom would show up downstream. (A leaked `-127` averaged with a real reading is *negative*, so it would never trip a high-temperature check — a monitor-level assertion would pass while the bug shipped.)

**Fill in the two `read()` methods below and run.**

In [ ]:
class GarbageOnFirstReadDevice:
    """Real device that returns 9999.9 if read before 50 ms have elapsed since creation."""
    def __init__(self, readings):
        self.readings = list(readings); self.i = 0
        self.created = time.time()
    def read(self):
        # TODO: if (time.time() - self.created) < 0.05, return 9999.9.
        #       Otherwise advance through self.readings (advance self.i, clamp at end).
        raise NotImplementedError('your turn')

class GlitchingDevice:
    """Real device whose firmware emits -127 on the second read; otherwise normal."""
    def __init__(self, readings):
        self.readings = list(readings); self.i = 0
    def read(self):
        # TODO: when self.i == 1, advance self.i and return -127.
        #       Otherwise advance through self.readings (advance self.i, clamp at end).
        raise NotImplementedError('your turn')

def test_adapters_absorb_real_hardware_failure_modes():
    # SensorA must wait out the warmup so the 9999.9 garbage never leaks through.
    a = SensorA(GarbageOnFirstReadDevice([42.0, 42.1, 42.2, 42.3]))
    a_readings = [a.read() for _ in range(4)]
    assert 9999.9 not in a_readings, f'SensorA leaked warmup garbage: {a_readings}'

    # SensorB must hold the last valid value when the firmware emits -127.
    b = SensorB(GlitchingDevice([43.0, 43.1, 43.2, 43.3]))
    b_readings = [b.read() for _ in range(4)]
    assert -127 not in b_readings, f'SensorB leaked -127 sentinel: {b_readings}'
    assert b_readings[1] == 43.0, f'SensorB should hold 43.0 over the glitch, got: {b_readings}'

    print('SIL passed — SensorA absorbed warmup garbage:', a_readings)
    print('SIL passed — SensorB absorbed -127 glitch:   ', b_readings)

test_adapters_absorb_real_hardware_failure_modes()

<details>
<summary><b>Reference solution</b> — try the exercise above before clicking</summary>
<pre><code class="language-python">class GarbageOnFirstReadDevice:
    def __init__(self, readings):
        self.readings = list(readings); self.i = 0
        self.created = time.time()
    def read(self):
        if time.time() - self.created < 0.05:
            return 9999.9
        v = self.readings[min(self.i, len(self.readings) - 1)]
        self.i += 1
        return v

class GlitchingDevice:
    def __init__(self, readings):
        self.readings = list(readings); self.i = 0
    def read(self):
        if self.i == 1:
            self.i += 1
            return -127
        v = self.readings[min(self.i, len(self.readings) - 1)]
        self.i += 1
        return v
</code></pre>
<p>The pattern: <em>each failure mode you learned about on hardware becomes a SIL fake that reproduces it deterministically, plus an assertion that the adapter strips it.</em> Now anyone editing <code>SensorA.read</code> or <code>SensorB.read</code> can't quietly remove the absorption — the assertion will catch it on the next commit.</p>
</details>

## Takeaway

The adapter pattern only matters if it's *defended*. The fakes you just wrote convert tribal knowledge ("sensor A needs warmup," "sensor B's firmware glitches") into runnable code that fires on every CI build.

The loop the whole progressive-testing lesson has been building toward:

- **Hardware** teaches you a failure mode you didn't imagine.
- The fix goes in the **adapter**, not the monitor — so the core stays clean.
- The failure mode goes in a **SIL fake**, not a tribal note — so the next regression is caught in milliseconds.

A rule of thumb worth keeping: if a fix mentions a hardware revision, a firmware version, a vendor ticket, or a specific serial number, it doesn't belong in the planner, the controller, or the monitor. It belongs at the interface, behind a name that says what it is — *and* behind a SIL fixture that proves it stays there.